In [ ]:
# Generic Imports
import numpy as np 
from PIL import Image
import scipy.io as sio 
import matplotlib.pyplot as plt 

In [ ]:
# ── Constants ──────────────────
RESIZE: int = 256
DATA_PATH: str = "./../../data/WillowObject/WILLOW-ObjectClass/"

In [ ]:
# ── Data Class ─────────────────
class NotatedImage:
    # 1. Constructor Method
    def __init__(self, img, kpts) -> None:
        self.img: Image.Image = img 
        self.kpts: np.array = kpts

    # 2. Resize Method
    def resize(self):
        self.kpts[0] *= RESIZE / self.img.size[0]
        self.kpts[1] *= RESIZE / self.img.size[1]
        self.img = self.img.resize((RESIZE, RESIZE), resample=Image.BILINEAR)
        return self

In [ ]:
# Local Imports
import glob # For searching files
import os   # To remove file extension

# ── Path Retrieval Function ──
def getting(cat: str, n: int) -> list[NotatedImage]:
    images: list[str] = glob.glob(f"{DATA_PATH}{cat}/*.png")
    points: list[str] = glob.glob(f"{DATA_PATH}{cat}/*.mat")
    output = []
    set_points = set(points)

    for img_file in images:
        base, _ = os.path.splitext(img_file)
        mat_file = base + ".mat"
        if mat_file in set_points:
            img = Image.open(img_file)
            kpts = np.array(sio.loadmat(mat_file)['pts_coord'])
            ni = NotatedImage(img, kpts)
            output.append(ni)
            
            if n is not None and len(output) >= n:
                break

    return output

In [ ]:
# Local Import
from scipy.spatial import Delaunay

# ── Delaunay Function ─────────
def delaunay_graph(self):
    points = list(zip(self.kpts[0], self.kpts[1]))
    tri = Delaunay(points)
    edges = []

    for simplex in tri.simplices:
        for i in range(len(simplex)):
            for j in range(i + 1, len(simplex)):
                edges.append((simplex[i], simplex[j]))

    self.edges = list(set(tuple(sorted(e)) for e in edges))
    return self

# We append it to NotatedImage
NotatedImage.add_delaunay = delaunay_graph

In [ ]:
import numpy as np
import networkx as nx

def make_adj(self):
    n = self.kpts.shape[1]
    self.adj = np.zeros((n, n), dtype=int)
    for i, j in self.edges:
        self.adj[i, j] = 1
        self.adj[j, i] = 1 # Undirected
    return self

NotatedImage.make_adj = make_adj

In [ ]:
class Pair:
    def __init__(self, ni_a, ni_b) -> None:
        self.ni_a = ni_a 
        self.ni_b = ni_b 

In [ ]:

# ── Hyperparameters ───────────────────────────────────────────────────────────
BETA_0    = 0.5
BETA_F    = 10.0
BETA_RATE = 1.075
I0        = 4
I1        = 30
EPS       = 0.5
OBJ_RESIZE = (256, 256)

# ── Provided utilities ────────────────────────────────────────────────────────
def visualize_matching_full(img1, img2, kpts1, kpts2, adj1, adj2, matching):
    """Visualize matching: yellow edges, green correct, red incorrect."""
    pass

In [ ]:
# ── Sinkhorn ──────────────────────────────────────────────────────────────
def sinkhorn(M, max_iter=30, eps=1e-3):
    """
    Alternate row/column normalization until M is doubly stochastic.

    Args:
        M        : (m×n) non-negative matrix
        max_iter : maximum iterations
        eps      : convergence threshold on ||M - M_old||₁

    Returns:
        M : doubly stochastic matrix (rows and columns sum to 1)

    Hints:
        - Normalize rows:    M = M / M.sum(axis=1, keepdims=True)
        - Normalize columns: M = M / M.sum(axis=0, keepdims=True)
        - Add 1e-10 in the denominator to avoid division by zero
        - Stop early if ||M - M_old||₁ < eps
    """
    
    for _ in range(max_iter):
        M_old = M.copy()
        # Added 1e-10 to avoid division by zero
        M = M / (M.sum(axis=1, keepdims=True) + 1e-10) 
        M = M / (M.sum(axis=0, keepdims=True) + 1e-10) 
        if np.abs(M - M_old).sum() < eps:
            break
    return M

In [ ]:
# ── Soft Assign ──────────────────────────────────────────────────────────────
def softassign(X_adj, Y_adj,
               beta0=BETA_0, beta_f=BETA_F, beta_rate=BETA_RATE,
               i0=I0, i1=I1, eps=EPS):
    """
    Graph matching by maximizing structural rectangles (Gold & Rangarajan 1996).

    Args:
        X_adj     : (m×m) adjacency matrix of G1
        Y_adj     : (n×n) adjacency matrix of G2

    Returns:
        M : (m×n) soft matching matrix

    Hints:
        - Initialize: M = ones(m,n) + 0.1*rand(m,n), then Sinkhorn
        - Outer loop: while beta < beta_f, multiply beta by beta_rate at the end
        - Inner loop (i0 iterations):
            · Q = X_adj @ M @ Y_adj
            · M = exp(beta * Q)
            · M = sinkhorn(M)
            · break if ||M - M_old||₁ < eps
    """
    
    m, n = X_adj.shape[0], Y_adj.shape[0]
    
    M = np.ones((m, n)) + 0.1 * np.random.rand(m, n)
    M = sinkhorn(M, max_iter=i1)
    beta = beta0
    
    while beta < beta_f:
        for _ in range(i0):
            M_old = M.copy()
            Q = X_adj @ M @ Y_adj
            M = np.exp(beta * Q)
            M = sinkhorn(M, max_iter=i1)
            
            if np.abs(M - M_old).sum() < eps:
                break 
        beta *= beta_rate

    return M


In [ ]:
# ── TO IMPLEMENT ──────────────────────────────────────────────────────────────

def cleanup(M_soft):
    """
    Convert soft matching to binary via greedy assignment.

    Args:
        M_soft : (m×n) soft matching matrix

    Returns:
        M_binary : (m×n) binary matching matrix

    Hints:
        - Repeatedly find argmax of M_work
        - Assign that entry, then set its row and column to -inf
    """
    len_x = M_soft.size[0]
    len_y = M_soft.size[1]

    M_binary = np.zeros(len_x, len_y)
    M_work = M_soft.copy() 
    
    closest = np.argmax(M_work, axis=1)
    
    for row, index in zip(M_binary, closest):
        row[index] = 1
    
    return M_binary

In [ ]:
# 1. Visualization
def visualize_matching_full(pa: Pair, save_path: str):
    img_a, img_b = pa.ni_a.img, pa.ni_b.img
    kpts_a, kpts_b = pa.ni_a.kpts, pa.ni_b.kpts
    
    width = max(img_a.size[0], img_b.size[0])
    height = max(img_a.size[1], img_b.size[1])
    composite = Image.new('RGB', (width * 2, height))
    composite.paste(img_a, (0, 0))
    composite.paste(img_b, (width, 0)) 
    
    plt.figure(figsize=(12, 6))
    plt.imshow(composite)
    plt.axis('off')
    
    for i, j in pa.ni_a.edges:
        plt.plot([kpts_a[0, i], kpts_a[0, j]], [kpts_a[1, i], kpts_a[1, j]], 'y-', alpha=0.5, lw=1)
    for i, j in pa.ni_b.edges:
        plt.plot([kpts_b[0, i] + width, kpts_b[0, j] + width], [kpts_b[1, i], kpts_b[1, j]], 'y-', alpha=0.5, lw=1)
        
    rows, cols = np.where(pa.match == 1)
    for i, j in zip(rows, cols):
        plt.plot([kpts_a[0, i], kpts_b[0, j] + width], [kpts_a[1, i], kpts_b[1, j]], 'g-', lw=1.5, alpha=0.8)
        
    plt.scatter(kpts_a[0], kpts_a[1], c='w', edgecolors='k', s=40, zorder=5)
    plt.scatter(kpts_b[0] + width, kpts_b[1], c='w', edgecolors='k', s=40, zorder=5)
    
    plt.title(f"Graph Matching Results - Acc: {pa.acc:.2f}")
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.close()